# Bell State Preparation with Hadamard and CNOT Gates

---

## 1. Problem Statement
This notebook demonstrates the preparation and measurement of a two-qubit maximally entangled **Bell state** ($|\Phi^+\rangle$) using a **Hadamard gate** followed by a **Controlled-NOT (CNOT / CX) gate**.

The objective is to verify quantum entanglement and correlated measurement outcomes by constructing the circuit in Qiskit, executing a shot-based simulation with `AerSimulator`, and validating that only the correlated computational basis states $|00\rangle$ and $|11\rangle$ are observed.

---

## 2. Theoretical Concepts & Component Roles

### Key Quantum Concepts & Components
- **Hadamard Gate ($H$)**: A single-qubit quantum gate that transforms computational basis states into equal superpositions:
  $$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle, \quad H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle$$
  Mathematically represented by the unitary matrix:
  $$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$
  In this circuit, applying $H$ to qubit 0 places it into an equal superposition of $|0\rangle$ and $|1\rangle$.
- **CNOT / CX Gate**: A two-qubit controlled unitary operation that applies a Pauli-X (bit flip) to the target qubit if and only if the control qubit is in state $|1\rangle$:
  $$\text{CNOT}|c, t\rangle = |c, c \oplus t\rangle$$
  In matrix form:
  $$\text{CNOT} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}$$
  When applied to a control qubit in superposition, the CNOT gate entangles the two qubits.
- **Control and Target Qubits**:
  - **Control Qubit ($q_0$)**: Initialized to $|0\rangle$ and placed into superposition $|+\rangle$ by the Hadamard gate. Its state conditionally determines whether the target qubit is flipped.
  - **Target Qubit ($q_1$)**: Initialized to $|0\rangle$. It remains $|0\rangle$ for the $|0\rangle$ component of the control qubit and flips to $|1\rangle$ for the $|1\rangle$ component of the control qubit.
- **Bell State ($|\Phi^+\rangle$)**: The resulting two-qubit state is one of the four canonical maximally entangled Bell states:
  $$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$
  In this entangled state, the individual qubits cannot be described by independent statevectors. Measuring either qubit immediately collapses the other qubit to the exact same value.
- **Measurement**: Projective measurement in the computational $Z$-basis ($\{|0\rangle, |1\rangle\}$). It collapses the entangled superposition state into classical bits stored in classical registers $c_0$ and $c_1$. Note that Qiskit represents classical bitstrings in little-endian order ($c_1 c_0$).
- **Shot-Based Simulation**: Emulating quantum hardware execution by repeatedly sampling measurement outcomes across a finite number of statistical trials ("shots", exactly 1024) using Qiskit Aer's `AerSimulator`.

---

## 3. Circuit Walkthrough & Step-by-Step Logic

The circuit executes according to the following step-by-step progression:

1. **Circuit Creation**: A 2-qubit, 2-classical-bit quantum circuit (`QuantumCircuit(2, 2)`) is created, initialized in the ground state $|q_1 q_0\rangle = |00\rangle$.
2. **Superposition Generation (`qc.h(0)`)**: `qc.h(0)` places qubit 0 into an equal superposition of $|0\rangle$ and $|1\rangle$:
   $$|00\rangle \xrightarrow{H_0} \frac{|00\rangle + |01\rangle}{\sqrt{2}} = |0\rangle \otimes \left(\frac{|0\rangle + |1\rangle}{\sqrt{2}}\right)$$
3. **Controlled Entanglement (`qc.cx(0, 1)`)**: `qc.cx(0, 1)` uses qubit 0 as the control and qubit 1 as the target. When qubit 0 is $|0\rangle$, qubit 1 remains $|0\rangle$; when qubit 0 is $|1\rangle$, qubit 1 flips to $|1\rangle$.
4. **Bell State Formation**: The resulting ideal state is the Bell state:
   $$\frac{|00\rangle + |11\rangle}{\sqrt{2}}$$
5. **Simultaneous Measurement**: Both qubits are measured (`qc.measure(0, 0)` and `qc.measure(1, 1)`), projecting the quantum state into classical register bits $c_0$ and $c_1$.
6. **Execution via AerSimulator**: The circuit is simulated with `AerSimulator` using exactly **1024 shots** to gather empirical sampling statistics.
7. **Validation Logic**: The existing validation checks that the observed outcomes are restricted strictly to `00` and `11` (ensuring that no orthogonal `01` or `10` states occur).
8. **Probability Calculation**: The existing probability calculation computes each observed count divided by 1024, quantifying empirical state probabilities.

---

## 4. Quantum Circuit Implementation & Simulation
Construct the 2-qubit Bell state circuit, execute on `AerSimulator` for 1024 shots, validate that only `00` and `11` are observed, and compute empirical outcome probabilities.

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

qc.h(0)
qc.cx(0, 1)

qc.measure(0, 0)
qc.measure(1, 1)

print(qc.draw())

simulator = AerSimulator()

job = simulator.run(qc, shots=1024)
result = job.result()
counts = result.get_counts()

print(counts)

valid_results = set(counts.keys())

if valid_results.issubset({'00', '11'}):
    print("SUCCESS!")
    print("Only 00 and 11 outcomes were observed.")
else:
    print("Unexpected outcomes:", valid_results)

for state, count in counts.items():
    probability = count / 1024
    print(state, "=", probability)

/var/folders/yp/078pyxw11mq6g7sd_2ty18z80000gn/T/ipykernel_2029/1513334865.py:1: DeprecationWarning: Using Qiskit with Python 3.9 is deprecated as of the 2.1.0 release. Support for running Qiskit with Python 3.9 will be removed in the 2.3.0 release, which coincides with when Python 3.9 goes end of life.
  from qiskit import QuantumCircuit


     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
{'11': 510, '00': 514}
SUCCESS!
Only 00 and 11 outcomes were observed.
11 = 0.498046875
00 = 0.501953125


---

## 5. Results Analysis & Conclusion

### Measurement Analysis & Ideal Expectations
- **Theoretical Expectations**: For an ideal Bell-state circuit preparing $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$, `00` and `11` are the expected measurement outcomes, with theoretical probabilities approaching **0.5 (50%) each** as the number of shots increases:
  $$P(00) = \left|\frac{1}{\sqrt{2}}\right|^2 = 0.5, \qquad P(11) = \left|\frac{1}{\sqrt{2}}\right|^2 = 0.5$$
- **Observed Notebook Results**:
  - Raw simulation counts: `{'11': 510, '00': 514}`
  - Computed empirical probabilities:
    - `11` = $510 / 1024 = 0.498046875$ (~49.80%)
    - `00` = $514 / 1024 = 0.501953125$ (~50.20%)
  - Validation output: `SUCCESS! Only 00 and 11 outcomes were observed.`
  - The slight divergence from exactly 0.5 is expected statistical fluctuation (shot noise) inherent to finite-shot sampling.

### Conclusion
This circuit demonstrates how a **Hadamard gate followed by CNOT** can create a two-qubit entangled Bell state whose computational-basis measurements are correlated. The complete absence of `01` and `10` outcomes verifies that measuring one qubit deterministically predicts the state of the other.